In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
import os
import shutil
import subprocess
import sys
import time

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    _t0 = time.time()

    def _el():
        return f"{time.time() - _t0:6.1f}s"

    # Write a placeholder submission before anything risky runs -- same
    # insurance pattern as the hypothesis-agent notebook (rules.md: a
    # submission is "auto-generated as long as the agent acts on the
    # games", so a total setup crash before that would otherwise leave no
    # submission file at all).
    import pandas as pd
    pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet("/kaggle/working/submission.parquet", index=False)
    print(f"[{_el()}] === step: wrote placeholder submission.parquet ===", flush=True)

    # Kaggle's own guidance: a code-competition rerun errors out if the agent
    # doesn't make its first move (open the environments) within roughly 15
    # minutes of container start. The official reference notebook
    # (arcprize/ARC-AGI-3-Kaggle-Starter) blocks on this exact curl check
    # with retry-max-time=600 *before* any of its own setup runs at all, and
    # that 600s figure is the one value actually proven against the real
    # gateway sidecar's startup latency. An earlier version of this cell
    # shrunk that to 90s to save time -- but main.py's own /api/games check
    # has no retry loop of its own (confirmed by reading main.py directly),
    # so if the real gateway ever takes longer than 90s under real
    # competition load, that shortened window would silently hand main.py an
    # unready gateway with no way to recover, on a path no free diagnostic
    # push can ever exercise to disprove. Restored to the official's proven
    # 600s -- the only thing kept from the earlier optimization is running
    # the wait as a background subprocess so our own file-copying/import
    # setup (which doesn't depend on the gateway at all) happens
    # *concurrently* with the wait instead of strictly after it, then joins
    # on it right before main.py needs the gateway to be up -- strictly
    # faster than the official's sequential version at the same worst-case
    # wait, never slower.
    print(f"[{_el()}] === step: starting gateway wait in background ===", flush=True)
    gateway_proc = subprocess.Popen(
        ["curl", "--fail", "--retry", "999", "--retry-all-errors",
         "--retry-delay", "2", "--retry-max-time", "600",
         "http://gateway:8001/api/games"],
    )

    print(f"[{_el()}] === step: checking numpy availability ===", flush=True)
    # GraphExplorerAgent is a pure classical-CV + graph-search agent -- no
    # torch, no learned model, no GPU dependency at all (kernel-metadata.json
    # sets enable_gpu=false accordingly). Only numpy is required beyond the
    # stdlib, and it ships in Kaggle's base image.
    import numpy
    print(f"[{_el()}] numpy {numpy.__version__} available", flush=True)

    print(f"[{_el()}] === step: copying competition harness repo (excluding .git) ===", flush=True)
    shutil.copytree(
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents",
        "/kaggle/working/ARC-AGI-3-Agents",
        ignore=shutil.ignore_patterns(".git"),
    )

    print(f"[{_el()}] === step: copying graph-explorer agent files ===", flush=True)
    # A Kaggle dataset_sources attachment mounts at
    # /kaggle/input/datasets/<owner>/<slug>/, not /kaggle/input/<slug>/ --
    # see CLAUDE.md's "Kaggle competition submission" section for the full
    # story of how this cost a whole debugging session the first time.
    _dataset_root = "/kaggle/input/datasets/calamitychasm/graph-explorer-agent"
    _templates_dir = "/kaggle/working/ARC-AGI-3-Agents/agents/templates"
    shutil.copy(f"{_dataset_root}/graph_explorer_core.py", f"{_templates_dir}/graph_explorer_core.py")
    shutil.copy(f"{_dataset_root}/graph_explorer_agent.py", f"{_templates_dir}/graph_explorer_agent.py")

    print(f"[{_el()}] === step: verifying copied files exist ===", flush=True)
    for p in [
        f"{_templates_dir}/graph_explorer_core.py",
        f"{_templates_dir}/graph_explorer_agent.py",
    ]:
        assert os.path.exists(p), f"missing expected file: {p}"
    print(f"[{_el()}] all expected files present", flush=True)

    print(f"[{_el()}] === step: writing agents/__init__.py ===", flush=True)
    # Minimal __init__.py -- avoids eagerly importing every other template
    # (langgraph/smolagents/etc, unmet deps in this image) and registers
    # only what we need.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write(
            "from typing import Type, cast\n"
            "from dotenv import load_dotenv\n"
            "from .agent import Agent, Playback\n"
            "from .swarm import Swarm\n"
            "from .templates.random_agent import Random\n"
            "from .templates.graph_explorer_agent import GraphExplorerAgent\n"
            "\n"
            "load_dotenv()\n"
            "\n"
            "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
            "    \"random\": Random,\n"
            "    \"graphexploreragent\": GraphExplorerAgent,\n"
            "}\n"
        )

    print(f"[{_el()}] === step: sanity-importing our agent before running main.py ===", flush=True)
    sys.path.insert(0, "/kaggle/working/ARC-AGI-3-Agents")
    sys.path.insert(0, "/kaggle/working")
    from agents.templates.graph_explorer_agent import GraphExplorerAgent  # noqa: F401
    print(f"[{_el()}] agent import OK", flush=True)

    print(f"[{_el()}] === step: writing .env ===", flush=True)
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write(
            "SCHEME=http\n"
            "HOST=gateway\n"
            "PORT=8001\n"
            "ARC_API_KEY=test-key-123\n"
            "ARC_BASE_URL=http://gateway:8001/\n"
            "OPERATION_MODE=online\n"
            "ENVIRONMENTS_DIR=\n"
            "RECORDINGS_DIR=/kaggle/working/server_recording\n"
        )

    print(f"[{_el()}] === step: joining background gateway-wait ===", flush=True)
    gw_rc = gateway_proc.wait()
    print(f"[{_el()}] gateway curl exited with code {gw_rc}", flush=True)
    if gw_rc != 0:
        print(f"[{_el()}] WARNING: gateway did not respond within the retry "
              f"window -- proceeding to main.py anyway (it may still come up moments "
              f"later); if this run errors, this line is the first thing to check.",
              flush=True)

    print(f"[{_el()}] === step: running agent ===", flush=True)
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "graphexploreragent"],
        cwd="/kaggle/working/ARC-AGI-3-Agents",
        env={**os.environ, "MPLBACKEND": "agg"},
    )
    print(f"[{_el()}] === main.py exited with code {result.returncode} ===", flush=True)


In [ ]:
# Non-rerun mode: produce a dummy submission
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
